# VoiceForge — Talking-head VIDEO backend

Turns one photo + your narration into a lip-synced MP4, using **SadTalker** (lip sync + blinking + head motion — best for short clips) and **Wav2Lip** (mouth-only, fast enough for long narration). Exposes a public HTTPS API via a Cloudflare tunnel.

**This is a SEPARATE backend from the TTS notebook.** Run it in its own session, after your speech is done, so the single free GPU is reused rather than needed twice at once.

**Before running:**
- **Colab** — Runtime → Change runtime type → **T4 GPU** → Save.
- **Kaggle** — session options sidebar → Accelerator → **GPU T4 x2**, and **Internet → On**.

Then **Run all**, top to bottom, in one pass. **No restart needed** — nothing is version-pinned any more; the few library incompatibilities are patched directly in the source instead.

Cell 3 prints the backend URL — paste it into the VoiceForge **Video** tab. Keep this tab open while rendering.

> Cell 1 prints a wall of red `pip` dependency-conflict lines about jax, opencv, rasterio, cupy and friends. **Those are expected and harmless** — they're Colab/Kaggle's preinstalled data-science stack, which this backend never imports.

In [ ]:
# Cell 1 — Install (SadTalker + Wav2Lip). ~8–12 min the first time.
import os, site, textwrap

# Do NOT downgrade numpy. Colab/Kaggle images are built around numpy 2 —
# opencv, scipy, tifffile and friends all require >=2, so forcing 1.26 fights
# the whole stack and doesn't even stick. SadTalker has exactly ONE numpy-2
# incompatibility, which we patch below instead.
!pip install -q fastapi uvicorn nest-asyncio python-multipart soundfile imageio imageio-ffmpeg
!pip install -q facexlib gfpgan basicsr insightface onnxruntime moviepy \
  yacs kornia pydub librosa numba resampy scikit-image tqdm safetensors av face-alignment
# batch_face = the RetinaFace detector the Wav2Lip fork imports.
!pip install -q batch-face

# --- SadTalker + checkpoints
if not os.path.exists('/content/SadTalker'):
    !git clone -q https://github.com/OpenTalker/SadTalker /content/SadTalker
!cd /content/SadTalker && bash scripts/download_models.sh

# --- Wav2Lip (justinjohn0306 fork: newer-torch friendly + hosts checkpoints)
if not os.path.exists('/content/Wav2Lip'):
    !git clone -q https://github.com/justinjohn0306/Wav2Lip /content/Wav2Lip
!mkdir -p /content/Wav2Lip/checkpoints /content/Wav2Lip/face_detection/detection/sfd

# The fork's inference.py hardcodes checkpoints/mobilenet.pth for RetinaFace,
# so grab every checkpoint the release ships.
W2L_REL = 'https://github.com/justinjohn0306/Wav2Lip/releases/download/models'
for name in ['wav2lip.pth', 'wav2lip_gan.pth', 'mobilenet.pth', 'resnet50.pth']:
    !wget -q -O /content/Wav2Lip/checkpoints/{name} {W2L_REL}/{name}
!wget -q -O /content/Wav2Lip/face_detection/detection/sfd/s3fd.pth {W2L_REL}/s3fd.pth

# A 404 leaves a 0-byte file that fails much later and cryptically. Check now.
print('\nWav2Lip checkpoints:')
for p in ['checkpoints/wav2lip_gan.pth', 'checkpoints/mobilenet.pth',
          'face_detection/detection/sfd/s3fd.pth']:
    full = f'/content/Wav2Lip/{p}'
    mb = os.path.getsize(full) / 1e6 if os.path.exists(full) else 0
    print(f'  {"OK " if mb > 1 else "BAD"} {p}: {mb:.1f} MB')

# ---------------------------------------------------------------------------
# Compatibility patches — applied to source, so no restart and no numpy pin.
# ---------------------------------------------------------------------------

# 1. numpy 2 removed np.VisibleDeprecationWarning (SadTalker's only use of a
#    removed numpy attribute — verified against the repo).
!grep -rl "np.VisibleDeprecationWarning" /content/SadTalker --include=*.py 2>/dev/null \
  | xargs -r sed -i 's/np\.VisibleDeprecationWarning/DeprecationWarning/g'

# 2. Newer torchvision removed transforms.functional_tensor, which basicsr and
#    facexlib still import.
!grep -rl "torchvision.transforms.functional_tensor" /usr/local/lib/python3*/dist-packages/ 2>/dev/null \
  | xargs -r sed -i 's/torchvision\.transforms\.functional_tensor/torchvision.transforms.functional/g'

# 3. torch >= 2.6 flipped torch.load to weights_only=True, which refuses these
#    pickled checkpoints. Both repos call torch.load in ~18 places, and they run
#    as subprocesses — so patch the default globally via sitecustomize, guarded
#    so it can never break interpreter startup.
_sc = textwrap.dedent('''
    try:
        import torch, functools
        if not getattr(torch.load, "_vf_patched", False):
            _orig = torch.load
            @functools.wraps(_orig)
            def _load(*a, **k):
                k.setdefault("weights_only", False)
                return _orig(*a, **k)
            _load._vf_patched = True
            torch.load = _load
    except Exception:
        pass
''')
for d in set(site.getsitepackages() + [site.getusersitepackages()]):
    try:
        os.makedirs(d, exist_ok=True)
        with open(os.path.join(d, 'sitecustomize.py'), 'w') as f:
            f.write(_sc)
    except Exception:
        pass
print('Patches applied (numpy 2, torchvision, torch.load).')

# --- Cloudflare tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

print('=' * 60)
print('Installed. Run Cell 2, then Cell 3 — no restart needed.')
print('(Ignore the red pip conflict lines about jax/opencv/rasterio etc.')
print(' The backend never imports those packages.)')
print('=' * 60)

In [ ]:
# Cell 2 — Verify GPU + report which engines are ready
#
# Nothing here is fatal except a missing GPU. An engine that isn't set up must
# never stop the server from starting — Cell 3 still gives you a URL, and the
# app hides whichever engine isn't available.
import torch, numpy as np, os, glob
assert torch.cuda.is_available(), 'No GPU. Colab: Runtime > Change runtime type > T4 GPU. Kaggle: session options > Accelerator > GPU.'
GPU = torch.cuda.get_device_name(0)
print(GPU, '| torch', torch.__version__, '| numpy', np.__version__)

def _have(path, min_mb=1):
    return os.path.exists(path) and os.path.getsize(path) / 1e6 >= min_mb

# SadTalker: checkpoints downloaded, and the numpy-2 patch actually applied.
SAD_FILES = bool(glob.glob('/content/SadTalker/checkpoints/*'))
_pp = '/content/SadTalker/src/face3d/util/preprocess.py'
SAD_PATCHED = (not os.path.exists(_pp)) or \
    ('np.VisibleDeprecationWarning' not in open(_pp).read())
SAD_OK = SAD_FILES and SAD_PATCHED

# Wav2Lip: every file the fork's inference.py actually touches.
W2L_FILES = [
    '/content/Wav2Lip/checkpoints/wav2lip_gan.pth',
    '/content/Wav2Lip/checkpoints/mobilenet.pth',
    '/content/Wav2Lip/face_detection/detection/sfd/s3fd.pth',
]
W2L_OK = all(_have(p) for p in W2L_FILES)

print('sadtalker ready:', SAD_OK)
if not SAD_OK:
    if not SAD_FILES:
        print('   checkpoints missing — re-run Cell 1, watch download_models.sh')
    if not SAD_PATCHED:
        print('   numpy-2 patch did not apply — re-run Cell 1')
print('wav2lip ready:  ', W2L_OK)
if not W2L_OK:
    for p in W2L_FILES:
        if not _have(p):
            print('   missing/empty:', p)

if not (SAD_OK or W2L_OK):
    print('\nNeither engine is set up. Cell 3 will still start and print a URL,')
    print('but rendering will fail until Cell 1 completes cleanly.')
else:
    print('\nRun Cell 3 for your backend URL.')

In [ ]:
# Cell 3 — API server + public URL
import io, os, base64, subprocess, threading, time, re, glob, shutil, uuid
import soundfile as sf, torch, nest_asyncio, uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'],
                   allow_methods=['*'], allow_headers=['*'])

WORK = '/content/vf_video'; os.makedirs(WORK, exist_ok=True)

def _engines():
    e = []
    if SAD_OK: e.append('sadtalker')
    if W2L_OK: e.append('wav2lip')
    return e

def _run(cmd, cwd):
    p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError((p.stderr or p.stdout or 'unknown error')[-1600:])

def to_wav(src_path, dst_path):
    """Both engines need real 16k mono WAV. The app can send mp3/m4a/webm
    (uploaded audio) as well as our own WAV, so always transcode."""
    p = subprocess.run(
        ['ffmpeg', '-y', '-i', src_path, '-ac', '1', '-ar', '16000', dst_path],
        capture_output=True, text=True)
    if p.returncode != 0 or not os.path.exists(dst_path):
        raise RuntimeError('Could not decode the audio: ' + (p.stderr or '')[-800:])
    return dst_path

def run_sadtalker(image_path, audio_path, out_dir):
    # --still reduces head jitter; --preprocess full keeps the whole photo
    # framed (vs 'crop' which zooms to the face); gfpgan sharpens the result.
    _run(['python', 'inference.py', '--driven_audio', audio_path,
          '--source_image', image_path, '--result_dir', out_dir,
          '--still', '--preprocess', 'full', '--enhancer', 'gfpgan'],
         cwd='/content/SadTalker')
    mp4s = sorted(glob.glob(os.path.join(out_dir, '**', '*.mp4'), recursive=True),
                  key=os.path.getmtime)
    if not mp4s: raise RuntimeError('SadTalker produced no mp4')
    return mp4s[-1]

def run_wav2lip(image_path, audio_path, out_path):
    _run(['python', 'inference.py', '--checkpoint_path', 'checkpoints/wav2lip_gan.pth',
          '--face', image_path, '--audio', audio_path, '--outfile', out_path,
          '--nosmooth', '--pads', '0', '15', '0', '0'],
         cwd='/content/Wav2Lip')
    if not os.path.exists(out_path): raise RuntimeError('Wav2Lip produced no mp4')
    return out_path

@app.get('/health')
def health():
    return {'ok': True, 'gpu': GPU, 'engines': _engines()}

@app.post('/animate')
async def animate(payload: dict):
    engine = payload.get('engine', 'sadtalker')
    if engine not in _engines():
        return JSONResponse({'error': f'{engine} not loaded'}, status_code=400)
    job = os.path.join(WORK, uuid.uuid4().hex[:10]); os.makedirs(job, exist_ok=True)
    try:
        mime = payload.get('image_mime', '')
        ext = 'jpg' if ('jpeg' in mime or 'jpg' in mime) else 'png'
        img = os.path.join(job, f'face.{ext}')
        raw = os.path.join(job, 'audio_raw')
        wav = os.path.join(job, 'audio.wav')
        with open(img, 'wb') as f: f.write(base64.b64decode(payload['image_b64']))
        with open(raw, 'wb') as f: f.write(base64.b64decode(payload['audio_b64']))
        to_wav(raw, wav)
        info = sf.info(wav); dur = info.frames / info.samplerate
        t0 = time.time()
        if engine == 'sadtalker':
            mp4 = run_sadtalker(img, wav, os.path.join(job, 'out'))
        else:
            mp4 = run_wav2lip(img, wav, os.path.join(job, 'out.mp4'))
        with open(mp4, 'rb') as f:
            b64 = base64.b64encode(f.read()).decode()
        return {'video_b64': b64, 'duration': round(dur, 2),
                'gen_seconds': round(time.time() - t0, 2)}
    except Exception as e:
        return JSONResponse({'error': str(e)[-1600:]}, status_code=500)
    finally:
        shutil.rmtree(job, ignore_errors=True)
        torch.cuda.empty_cache()

# --- start server + tunnel ---
nest_asyncio.apply()
threading.Thread(
    target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000, log_level='error'),
    daemon=True,
).start()
time.sleep(3)

proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    m = re.search(r'https://[-\w]+\.trycloudflare\.com', line)
    if m:
        print('\n' + '=' * 60)
        print('  VIDEO BACKEND URL — paste into the VoiceForge Video tab:')
        print('  ' + m.group(0))
        print('=' * 60 + '\n')
        print('Engines available:', _engines())
        print('Keep this tab open while rendering.')
        break

In [ ]:
# Cell 4 — Keep-alive (optional)
# Colab disconnects idle tabs. Run this in the background during long renders.
import time
while True:
    time.sleep(60)
    print('.', end='', flush=True)